<img src="../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<img src="assets/agent09-top-light.svg" alt="A pretrained model is fine-tuned on your data and becomes the teacher; the teacher's labels go to a Zarr store; a small, fast student is distilled from them and then runs the simulation." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 5px solid #76b900; border-radius: 4px; padding: 24px 28px; margin: 8px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="font-variant: small-caps; font-size: 12px; font-weight: 700; letter-spacing: 2px; opacity: 0.75;">AGENTIC TRACK, NOTEBOOK 09</div><h1 style="margin: 6px 0 8px; padding: 0; border: 0;"><span style="font-size: 32px; font-weight: 700; line-height: 1.2; color: inherit;">Small-batch distilled</span></h1><div style="font-size: 18px; line-height: 1.45; opacity: 0.85;">Build a dataset and distill a small, fast model for your task.</div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 12px 18px; margin: 12px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; line-height: 1.6;"><b>Hardware:</b> a CUDA-capable NVIDIA GPU<br><b>Prerequisites:</b> Meet your agent; Many molecules, one batch; Ready-made; Hooked in at every step<br><b>References:</b> <a href="https://nvidia.github.io/nvalchemi-toolkit/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit documentation</a>; <a href="https://nvidia.github.io/nvalchemi-toolkit/examples/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit examples</a>; <a href="https://github.com/NVIDIA/skills/tree/main/skills/nvalchemi-toolkit" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">the nvalchemi-toolkit skill</a></div>

<img src="assets/agent09-roadmap-light.svg" alt="Sections of this notebook on one line." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

MACE-MPA-0 learned mostly from crystals at and near their relaxed structures. Liquid SiC at
3000 to 4000 K lies outside that: on 60 frames of liquid, quenched and amorphous SiC at a density
left out of the fine-tune, the model is off by about 63 meV/atom in energy and 0.46 eV/Å in force.
A few hundred DFT frames lower both errors: fine-tuning.
Long MD of the liquid then needs speed more than it needs a model of 9 million weights, so a small
student learns the fine-tuned model's forces on this liquid alone: distillation. Toolkit has both
training loops, and the sampling, labelling and Zarr storage between them.

In [ ]:
# Load the course helpers, the liquid SiC frames and the pretrained model
import torch
from helpers import agent09 as helpers
helpers.configure_tutorial()             # course display defaults
work_dir = helpers.make_work_dir()       # notebooks/agent09_work
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_path, test_path, base_model = helpers.prepare_finetuning(device)   # DFT frames in two cuts; MACE-MPA-0
ft_steps, n_replicas, md_steps, train_steps, test_steps = 120, 16, 400, 1500, 800   # optimizer steps; copies, MD steps, training steps, MD steps of the student's run
starts, test_starts = helpers.load_starts(n_replicas)   # liquid cells for the teacher's MD and for the student's
helpers.show_session()                   # prints what your agent needs to start

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">1. Fine-tune a pretrained model</h2>

`train_path` holds 300 DFT frames of liquid SiC, 216 atoms each, at five densities:
the melting run, the equilibrated liquid, the quench and the amorphous solid of each.
`test_path` holds the 60 frames of a sixth density, 2.7 g/cm³, which the training never sees.
The frames are from the SiC-TGAP set of
[Hamedani and Sand (2025)](https://doi.org/10.5281/zenodo.17182227) (PBE, CC BY 4.0). Start
with the model as published; the loader has only moved its energy reference to that of the set.

In [ ]:
# Score the published model on the held-out frames
helpers.show_errors(test_path, base_model)

Each point is one force component on one atom of a held-out frame: the
model's value against the DFT value. The table gives the root mean square
errors for each kind of frame and over all of them.

#### Example: fine-tune on the training frames

`FineTuningStrategy` trains the weights you name and keeps the rest as
published. Train the last interaction block and the readouts of a copy of
`base_model`, so that `base_model` itself stays as published. Use AdamW with a
learning rate of 5e-4, batches of 4 frames, `ft_steps` optimizer steps, and a
loss of the energy per atom plus 0.1 times the forces.

| | |
| --- | --- |
| input  | `base_model`; `train_path`; `ft_steps` |
| output | `finetuned_model`, `ft_loss_history`, in `agent09_work/finetune.py` |

> 💡 **Hint**
>
> In MACE-MPA-0 the last interaction block is `interactions.1` with `products.1`;
> inside the strategy every weight name starts with `main.model.`.

In [ ]:
%load agent09_work/finetune.py

In [ ]:
# Plot the training loss, then score both models on the held-out frames
helpers.show_loss(ft_loss_history)
helpers.show_errors(test_path, base_model, finetuned_model)

The curve is the training loss on the frames of `train_path`. The table has
the rows of the first one, with the fine-tuned model beside the published one.

> </> API
>
> `FineTuningStrategy(models=..., trainable_patterns=("main.model.readouts.*", ...), ...)`
>
> **Input:** a pretrained model, name patterns for the weights to train<br>
> **Effect:** only the matching weights train, in the model you pass; a pattern that matches nothing raises `ValueError`

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">2. Distill a small student</h2>

The fine-tuned model is now the teacher. A student learns from structures
with an energy and forces for each. A hot MD of the liquid cells in `starts`
supplies them, each cell at the temperature of its DFT run; a hook saves the
frames to a Zarr store as the run goes, and `label_dataset` writes the
teacher's answers beside each one.

#### Try it: sample and label frames

Run the cells in `starts` with `finetuned_model` for `md_steps` steps of 1 fs, with a
friction of 0.01 per fs and a frame of every cell saved every 10 steps, then label the frames.

| | |
| --- | --- |
| input  | `starts`; `finetuned_model`; `md_steps` |
| output | `labeled_path`, in `agent09_work/sample_label.py` |

> 💡 **Hint**
>
> `helpers.show_store(labeled_path)` draws the result. `starts` from the setup cell is the
> tested default: one liquid cell per copy, with its temperature in `atoms.info["temperature_K"]`.
> `NVTLangevin` takes one temperature per copy as a tensor.

Done when: the table lists `teacher_energy` and `teacher_forces`, saved for every frame.

In [ ]:
%load agent09_work/sample_label.py

In [ ]:
# Show the labelled store
helpers.show_store(labeled_path)

Each row is one field of the store, saved for every frame.

> </> API
>
> `label_dataset(dataset, InProcessTeacherScorer(teacher, ("energy", "forces")), path)`
>
> **Input:** a dataset of structures, a teacher model, a path<br>
> **Effect:** writes a new store at the path, with `teacher_energy` and `teacher_forces` beside each structure

#### Example: distill the student

`helpers.make_student(labeled_path)` returns a small MACE model with
untrained weights, 34,464 of them against the teacher's 9.06 million.
`DistillationStrategy` trains it on the stored labels; the teacher's
weights stay as they are. Use a learning rate of 1e-2, batches of 4 frames,
and a loss of the energy per atom plus the forces.

| | |
| --- | --- |
| input  | `labeled_path`; `finetuned_model`; `helpers.make_student(labeled_path)`; `train_steps` |
| output | `student_model`, `loss_history`, in `agent09_work/distill.py` |

In [ ]:
%load agent09_work/distill.py

In [ ]:
# Plot the training loss against the step
helpers.show_loss(loss_history)

The curve is the training loss on the frames of the store: the student's
energy and forces against the teacher's labels.

> </> API
>
> `DistillationStrategy(models={"student": ..., "teacher": ...}, loss_fn=..., num_steps=...)`
>
> **Input:** a student, a teacher, a loss whose targets are the teacher's fields, such as `EnergyMSELoss(target_key="teacher_energy")`<br>
> **Effect:** only the student trains

> ⚙️ **Fine control**
>
> ```python
> OptimizerConfig(
>     optimizer_cls=torch.optim.AdamW,
>     optimizer_kwargs={"lr": ...},        # learning rate
> )
> ```
>
> Example: `train with a learning rate of 5e-3`

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">3. Simulate with the student</h2>

Now use the student for the job it was made for: hot MD of the liquid.
Toolkit's distillation guide measures a student on a trajectory the
student drives itself. Start it from `test_starts`, other frames of the same
DFT runs, so that the frames are new to the student.

#### Try it: run the MD with the student

Run the cells in `test_starts` with `student_model` for `test_steps` steps, with the
settings of the sampling run, and register a `StabilityMonitor` on the run.

| | |
| --- | --- |
| input  | `student_model`; `test_starts`; `test_steps` |
| output | `student_frames_path`, `stability`, in `agent09_work/run_student.py` |

> 💡 **Hint**
>
> `helpers.show_stability(student_frames_path, stability)` draws the result. `test_starts`
> from the setup cell is the tested default. `StabilityMonitor` is in `nvalchemi.dynamics.hooks`;
> `stability` is its `metrics()` after the run; its `first_divergence_step` is the first step at
> which a structure stopped being finite. `student_frames_path` is the path (`str` or
> `Path`) of a new Zarr store with a frame of every cell every 10 steps.

Done when: the table has a row for each density of `test_starts`.

In [ ]:
%load agent09_work/run_student.py

In [ ]:
# Check whether every copy is still liquid
helpers.show_stability(student_frames_path, stability)

One row per density. A copy is still liquid when its mean temperature stays
within 15% of the target, no two atoms come closer than 1.0 Å, and its atoms
keep moving apart: a mean squared displacement of at least 2 Å² at the end,
and at least 1.25 times its value at half time.

> </> API
>
> `StabilityMonitor(frequency=..., timestep_fs=...)`
>
> **Input:** how often to record, the time step in fs<br>
> **Effect:** a hook that records the run's energy and momentum; `metrics()` after the run gives the drift

#### Try it: compare the student's structure with the teacher's

Compute the radial distribution g(r) over all pairs, with Toolkit's
`radial_distribution` and the same bins for both: once for the teacher's
sampling run, the frames of `labeled_path`, and once for the student's run.

| | |
| --- | --- |
| input  | `labeled_path`; `student_frames_path` |
| output | `teacher_rdf`, `student_rdf`, in `agent09_work/structure.py` |

> 💡 **Hint**
>
> `helpers.show_structure(teacher_rdf, student_rdf, labeled_path, student_frames_path)` draws
> the result. `radial_distribution` is in `nvalchemi.training.distillation.evaluation`; the tested
> default is `radial_distribution(batch, r_max=6.0, num_bins=60)`, with `batch` a `Batch` of periodic
> frames. `helpers.read_store(path)` returns `(reader, batch)` with every frame of a store.

Done when: the figure shows both curves and the table gives their divergence.

In [ ]:
%load agent09_work/structure.py

In [ ]:
# Draw both radial distributions, over all pairs and for each pair of elements
helpers.show_structure(teacher_rdf, student_rdf, labeled_path, student_frames_path)

Grey is the teacher's sampling run, green the student's. The helper adds the
curve of each pair of elements from the same stores. The table gives the
Jensen-Shannon divergence of each pair of curves, 0 for identical ones, and
the position of each first peak.

In [ ]:
# The student against the teacher on its own frames, the three models against DFT, and the speed
helpers.show_student_vs_teacher(student_frames_path, finetuned_model)
helpers.show_errors(test_path, base_model, finetuned_model, student_model)
helpers.show_speed(finetuned_model, student_model, test_starts)

The plot sets the student's force on each atom of its own frames against
the teacher's. The table under it scores the three models against DFT on the
held-out density. The bars give MD steps per second with each model on the
same cells, timed after a warm-up; the line under them gives the number of runs.

The student is a fast model for the forces and structure of this liquid. Against
DFT its force error, about 0.43 eV/Å, sits between the published model's 0.46
and the teacher's 0.35. Its
energies sit off the teacher's by 20 to 70 meV/atom from one training run to
the next, and the offset changes from one region of the liquid to another.
Use it for forces and structure, not for energies.

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">Explore more</h2>

More ways to train with Toolkit, each a runnable example.

<div style="display: flex; flex-wrap: wrap; gap: 16px; margin: 4px 0 18px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/intermediate/08_lora_finetuning.py">LoRA fine-tuning</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">MACE-MPA-0 with low-rank adapters, saved and loaded again.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/intermediate/09_offline_distillation.py">Offline distillation</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">A teacher labels a store once; the student trains from it.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/intermediate/10_onpolicy_distillation.py">On-policy distillation</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">The student's own MD supplies the frames the teacher labels.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/10_mace_training.py">Train from scratch</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;"><code>TrainingStrategy</code> trains a MACE model from untrained weights.</div></div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; padding: 15px 20px; margin: 24px 0 18px; line-height: 1.6;"><h2 style="margin: 0 0 10px; padding: 0; border: 0; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 24px; font-weight: 500; line-height: 1.25; color: inherit;">Notebook 09 recap</h2><div style="margin: 0 0 10px;">You fine-tuned MACE-MPA-0 on 300 DFT frames of liquid SiC and lowered its energy and force errors at a density it never saw. The fine-tuned model sampled the hot liquid and labelled the frames, and a student of 34,464 weights learned from those labels. In its own run the student kept the liquid liquid, followed the teacher's structure and ran about 20 times faster; its energies carry an offset, so use it for forces and structure. A student knows only what its dataset showed it: check it on new structures before you rely on it.</div><div style="margin: 0 0 10px;"><b>This is the last notebook of the agentic track.</b> Bring your own systems, and keep checking every result against your domain knowledge.</div></div>